In [1]:
import pandas as pd
import re

In [6]:
base = pd.read_csv("../data/limpieza/observaciones_ti.csv", skiprows=[0, 2])

In [7]:
df = base.copy()

In [ ]:
df

Index(['Fecha de inicio', 'Fecha de finalización', 'Tipo de respuesta',
       'Dirección IP', 'Progreso', 'Duración (en segundos)', 'Finalizado',
       'Fecha registrada', 'ID de respuesta', 'Apellido del destinatario',
       ...
       '¿Están los/las estudiantes trabajando individualmente?.10',
       '¿Están los/las estudiantes trabajando en parejas o grupos?.11',
       '¿Quiénes están ejerciendo roles de liderazgo en el trabajo en parejas o grupos?.11',
       '¿Están los/las estudiantes realizando las actividades por sí mismos, sin ayuda del docente?.11',
       'Agregue cualquier comentario adicional, que sea relevante para entender lo que está sucediendo en el aula en este instante de la clase.11',
       'Timing - Primer clic.12', 'Timing - Último clic.12',
       'Timing - Envío de página.12', 'Timing - Recuento de clics.12',
       'Por favor, seleccione qué desea hacer.'],
      dtype='object', length=445)

In [4]:
df = df.loc[(df["Finalizado"] == True) | (df["Progreso"] > 80), :]

In [5]:
df = df.loc[df["Tipo de respuesta"] != "Survey Preview", :]

In [6]:
# Renombrar columnas que contienen "Colegios" a "Colegio"

df.columns = [col.replace("Colegios", "Colegio") for col in df.columns]
# Crear columnas auxiliares para mentor y colegio
df['col_mentor'] = "Mentores " + df['Unidad']
df['col_colegio'] = "Colegio " + df['Unidad']
# Extraer mentor y colegio usando las columnas auxiliares
df['Mentor'] = df.apply(lambda row: row[row['col_mentor']] if not pd.isna(row['col_mentor']) else None, axis=1)
df['colegio'] = df.apply(lambda row: row[row['col_colegio']] if not pd.isna(row['col_mentor']) else None, axis=1)
# Dividir la columna 'colegio' en 'Código IE' y 'Sede'
df["colegio"] = df['colegio'].apply(lambda x: x.split(" ") if x and isinstance(x, str) else ["", ""])
df["Código IE"] = df["colegio"].apply(lambda x: x[0] if x else None)
df["Sede"] = df["colegio"].apply(lambda x: " ".join(x[1:]) if x else None)
df.columns = [re.sub(r"\s+"," ", col.strip().replace("\n", " ")) for col in df.columns]

In [7]:
#- Visita 1 = [abril-mayo]
#- visita 2 = [junio-julio]
#- visita 3 = [sept-nov]
# Keep only latest row per visita per Número de documento de docente observado/a

visita_dates = [
    (1, "2025-04-01", "2025-05-31"),
    (2, "2025-06-01", "2025-07-31"),
    (3, "2025-09-01", "2025-11-30"),
]

# "19-03-2025" or "19/03/2025"
df["Fecha"] = df["Fecha del acompañamiento Indique la fecha en que realizó el acompañamiento en aula. Use el formato dd/mm/aaaa"].str.replace("/", "-")
df["Fecha"] = pd.to_datetime(df["Fecha"], format="%d-%m-%Y", errors="raise")
df = df[df['Fecha de inicio'] >= pd.to_datetime('2025-04-01')]
df["Visita"] = None


for visita, start_date, end_date in visita_dates:
    mask = (df["Fecha"] >= start_date) & (df["Fecha"] <= end_date)
    df.loc[mask, "Visita"] = visita

df = df.sort_values(by=["Número de documento de docente observado/a", "Fecha de finalización"], ascending=False)
df = df.drop_duplicates(subset=["Número de documento de docente observado/a", "Visita"], keep="first")


df.dropna(subset=['Visita'], inplace=True)

In [8]:
# Keep only last 2 per docente (Número de documento de docente observado/a), assign the column Momento (Pre o Post)
df = df.sort_values(by=["Número de documento de docente observado/a", "Fecha"], ascending=[True, False])
df = df.groupby("Número de documento de docente observado/a").head(2).reset_index(drop=True)
df = df.sort_values(by=["Número de documento de docente observado/a", "Fecha"], ascending=[True, True])
df["Momento"] = df.groupby("Número de documento de docente observado/a").cumcount().map({0: "Pre", 1: "Post"})

# Remove observations with only one Momento
df = df.groupby('Número de documento de docente observado/a').filter(lambda x: x['Momento'].nunique() > 1)

In [9]:
def concatenate_columns(df, start_col, end_col, target_col):
    """
    Concatena los valores de las columnas desde start_col hasta end_col
    y los asigna a la columna target_col en cada fila.

    Args:
        df (pandas.DataFrame): DataFrame de entrada.
        start_col (str): Nombre de la primera columna a concatenar.
        end_col (str): Nombre de la última columna a concatenar.
        target_col (str): Nombre de la columna donde se guardará la concatenación.

    Returns:
        pandas.DataFrame: DataFrame con la columna target_col actualizada.
    """
    # Verifica que las columnas existan
    if start_col not in df.columns or end_col not in df.columns:
        raise ValueError("Alguna de las columnas especificadas no existe en el DataFrame.")

    # Obtiene los índices de las columnas
    start_idx = df.columns.get_loc(start_col)
    end_idx = df.columns.get_loc(end_col)

    # Selecciona las columnas a concatenar
    columns_to_concat = df.columns[start_idx:end_idx + 1]

    # Convierte los valores a string y concatena por fila, manejando NaN
    df[target_col] = df[columns_to_concat].astype(str).agg(' '.join, axis=1).str.replace('nan', '', regex=False)

    return df

# Llamar a la función
df = concatenate_columns(df, 'Mentores UG1', 'Mentores UG24', 'Mentor')
df["Mentor"]


# Función para limpiar espacios innecesarios en los valores de una columna
def clean_column_values(df, column_name):
    """
    Limpia los valores de una columna eliminando espacios al principio y al final,
    y reemplazando múltiples espacios consecutivos por un solo espacio.

    Args:
        df (pandas.DataFrame): DataFrame de entrada.
        column_name (str): Nombre de la columna cuyos valores se van a limpiar.

    Returns:
        pandas.DataFrame: DataFrame con la columna limpiada.
    """
    # Verifica que la columna exista
    if column_name not in df.columns:
        raise ValueError(f"La columna '{column_name}' no existe en el DataFrame.")

    # Aplica la limpieza a cada valor de la columna
    df[column_name] = df[column_name].apply(lambda x: re.sub(r'\s+', ' ', str(x).strip()) if pd.notnull(x) else x)

    return df

df = clean_column_values(df, 'Mentor')


def drop_column_range(df, start_col, end_col):
    """
    Elimina un rango de columnas consecutivas desde start_col hasta end_col (inclusive).

    Args:
        df (pandas.DataFrame): DataFrame de entrada.
        start_col (str): Nombre de la primera columna a eliminar.
        end_col (str): Nombre de la última columna a eliminar.

    Returns:
        pandas.DataFrame: DataFrame con las columnas eliminadas.
    """
    # Verifica que las columnas existan
    if start_col not in df.columns or end_col not in df.columns:
        raise ValueError("Alguna de las columnas especificadas no existe en el DataFrame.")

    # Obtiene los índices de las columnas
    start_idx = df.columns.get_loc(start_col)
    end_idx = df.columns.get_loc(end_col)

    # Crea una lista de columnas a eliminar usando slicing
    columns_to_drop = df.columns[start_idx:end_idx + 1].tolist()

    # Elimina las columnas
    df_dropped = df.drop(columns=columns_to_drop)

    return df_dropped

df = drop_column_range(df, 'Mentores UG1', 'Mentores UG24')
df = drop_column_range(df, 'Colegio UG1', 'Colegio UG24')
df = drop_column_range(df, 'Apellido del destinatario', 'Referencia a datos externos')

In [10]:
copia = df.copy()

In [11]:
copia

,Fecha de inicio,Fecha de finalización,Tipo de respuesta,Dirección IP,Progreso,Duración (en segundos),Finalizado,Fecha registrada,ID de respuesta,Latitud de la ubicación,...,"Por favor, seleccione qué desea hacer.",col_mentor,col_colegio,Mentor,colegio,Código IE,Sede,Fecha,Visita,Momento
2,2025-06-04 11:45:40,2025-06-04 14:20:19,IP Address,191.95.36.249,100,9278,True,2025-06-04 14:20:20.753,R_7esDIa291bLHEUw,6.2529,...,NaN,Mentores UG20,Colegio UG20,Samuel Antonio Rua Londoño,"[NPC064, IE, Leticia, Arango, de, Avendaño]",NPC064,IE Leticia Arango de Avendaño,2025-06-04,2,Pre
1,2025-09-17 11:45:04,2025-09-17 12:58:19,IP Address,191.95.39.37,100,4394,True,2025-09-17 12:58:20.180,R_7r8EwEBEf2yeEhw,6.2529,...,NaN,Mentores UG20,Colegio UG20,Samuel Antonio Rua Londoño,"[NPC064, IE, Leticia, Arango, de, Avendaño]",NPC064,IE Leticia Arango de Avendaño,2025-09-17,3,Post
9,2025-04-29 15:00:59,2025-04-29 15:46:41,IP Address,223.27.115.82,100,2740,True,2025-04-29 15:46:41.957,R_1ffrXqDCHFeK9vi,4.9973,...,NaN,Mentores UG20,Colegio UG20,Mónica Katerine Cristancho Vega,"[NPC110, IE, Colegio, Guillermo, León, Valenci...",NPC110,IE Colegio Guillermo León Valencia - Principal...,2025-04-29,1,Pre
8,2025-09-24 16:22:36,2025-09-24 17:04:46,IP Address,190.102.122.94,100,2529,True,2025-09-24 17:04:46.857,R_3SjC2EpHxhXK6yk,4.3356,...,NaN,Mentores UG20,Colegio UG20,Mónica Katerine Cristancho Vega,"[NPC110, IE, Colegio, Guillermo, León, Valenci...",NPC110,IE Colegio Guillermo León Valencia - Principal...,2025-09-24,3,Post
11,2025-04-29 14:48:18,2025-04-29 16:13:47,IP Address,190.26.221.61,100,5129,True,2025-04-29 16:13:48.561,R_5qjsHomeq9XN5F7,4.6544,...,NaN,Mentores UG10,Colegio UG10,Estefanía Nieves Torres,"[NPC234, Cent, Educ, Dist, El, Porvenir]",NPC234,Cent Educ Dist El Porvenir,2025-04-29,1,Pre
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,2025-10-02 14:37:00,2025-10-02 15:45:58,IP Address,181.143.61.188,100,4137,True,2025-10-02 15:45:59.473,R_1ahGCVUVPVaxvGN,6.2529,...,NaN,Mentores UG18,Colegio UG18,Herdo Manrique,"[NPC071, IE, Pueblo, Nuevo]",NPC071,IE Pueblo Nuevo,2025-09-30,3,Post
800,2025-04-28 15:20:34,2025-04-28 15:49:56,IP Address,24.152.58.179,100,1761,True,2025-04-28 15:49:56.821,R_7nIaUt1Pf0Z2r9l,7.1274,...,NaN,Mentores UG1,Colegio UG1,Jose Gregorio Beltrán Garcé,"[, ]",,,2025-04-28,1,Pre
799,2025-10-01 17:06:10,2025-10-01 17:41:05,IP Address,45.172.223.140,100,2095,True,2025-10-01 17:41:06.254,R_1efRKcyfddxQ0y0,6.5557,...,NaN,Mentores UG1,Colegio UG1,Jose Gregorio Beltrán Garcé,"[NPC324, Escuela, Normal, Superior, Francisco,...",NPC324,Escuela Normal Superior Francisco de Paula San...,2025-10-01,3,Post
806,2025-05-03 11:01:04,2025-05-03 12:00:23,IP Address,181.52.61.22,100,3558,True,2025-05-03 12:00:24.329,R_7j6uWowM7Mi3LNj,1.2146,...,NaN,Mentores UG14,Colegio UG14,William Germán García Mora,"[NPC222, IE, Técnica, Industrial, Antonio, Jos...",NPC222,IE Técnica Industrial Antonio José Camacho,2025-04-29,1,Pre


In [12]:
lista_instantaneas = ['Instantánea 1 - minuto 8 de la observación ¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante.','¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante.', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..1', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..2', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..3', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..4', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..5', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..6', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..7', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..8', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..9', '¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..10']

columnas = [
    "¿Qué está haciendo el/la docente ahora?",
    "Indique con quiénes está interactuando el/la docente",
    "¿Con quiénes está interactuando el/la docente?",
    "¿Están los/las estudiantes respondiendo las preguntas y/o participando en las discusiones?",
    "¿Están los/las estudiantes escuchando atentamente al docente?",
    "¿Están los/las estudiantes tomando nota de las explicaciones o discusiones?",
    "¿Están los/las estudiantes haciendo preguntas y/o pidiendo ayuda al docente?",
    "¿Están los/las estudiantes socializando su trabajo?",
    "¿Quiénes están socializando su trabajo?",
    "¿Están los/las estudiantes distraídos, haciendo indisciplina o mostrando de alguna otra manera que no están involucrados en las actividades que lidera el/la docente?",
    "¿Quiénes no están involucrados en las actividades que lidera el/la docente?",
    "¿Están haciendo uso de herramientas computacionales?",
    "¿Quiénes están haciendo uso de herramientas computacionales?",
    "¿Están los/las estudiantes trabajando individualmente?",
    "¿Están los/las estudiantes trabajando en parejas o grupos?",
    "¿Quiénes están ejerciendo roles de liderazgo en el trabajo en parejas o grupos?",
    "¿Están los/las estudiantes realizando las actividades por sí mismos, sin ayuda del docente?",
    "Agregue cualquier comentario adicional, que sea relevante para entender lo que está sucediendo en el aula en este instante de la clase",
    'ID de respuesta',
    "Número de documento de docente observado/a",
    "Nombre de docente observado/a",
    "Indique sexo del docente",
    "¿El/la docente trabaja con una guía pedagógica?",
    "Información de la clase Asignatura - Selected Choice",
    "Código IE",
    "Visita",
    "Número de instantánea",  # Nueva primera columna,
]
instantaneas = pd.DataFrame()

i = 1
for instantanea in lista_instantaneas:
  print(instantanea)
  columna = df.columns.get_loc(instantanea)

  tablita = df.iloc[:,columna:columna+18].copy()
  tablita = tablita.rename(columns={instantanea:'¿Qué está haciendo el/la docente ahora?' })
  tablita = tablita.merge(df.loc[:,['ID de respuesta',"Número de documento de docente observado/a" ,'Nombre de docente observado/a', 'Indique sexo del docente', '¿El/la docente trabaja con una guía pedagógica?', 'Información de la clase Asignatura - Selected Choice','Código IE',"Visita"]], left_index=True, right_index=True)
  tablita.loc[:,'Número de instantánea'] = i
  tablita.columns = columnas
  instantaneas = pd.concat([instantaneas,tablita],axis=0)
  i=i+1

Instantánea 1 - minuto 8 de la observación ¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante.
¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante.
¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..1
¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..2
¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..3
¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante..4
¿Qué está haciendo el/la docente ahora? Indique cuál de las s

In [13]:
nuevo_orden = ['ID de respuesta',
 'Número de instantánea','Visita',
 'Número de documento de docente observado/a',
 'Nombre de docente observado/a',
 'Indique sexo del docente',
 '¿El/la docente trabaja con una guía pedagógica?',
 'Información de la clase Asignatura - Selected Choice',
 'Código IE','¿Qué está haciendo el/la docente ahora?',
 'Indique con quiénes está interactuando el/la docente',
 '¿Con quiénes está interactuando el/la docente?',
 '¿Están los/las estudiantes respondiendo las preguntas y/o participando en las discusiones?',
 '¿Están los/las estudiantes escuchando atentamente al docente?',
 '¿Están los/las estudiantes tomando nota de las explicaciones o discusiones?',
 '¿Están los/las estudiantes haciendo preguntas y/o pidiendo ayuda al docente?',
 '¿Están los/las estudiantes socializando su trabajo?',
 '¿Quiénes están socializando su trabajo?',
 '¿Están los/las estudiantes distraídos, haciendo indisciplina o mostrando de alguna otra manera que no están involucrados en las actividades que lidera el/la docente?',
 '¿Quiénes no están involucrados en las actividades que lidera el/la docente?',
 '¿Están haciendo uso de herramientas computacionales?',
 '¿Quiénes están haciendo uso de herramientas computacionales?',
 '¿Están los/las estudiantes trabajando individualmente?',
 '¿Están los/las estudiantes trabajando en parejas o grupos?',
 '¿Quiénes están ejerciendo roles de liderazgo en el trabajo en parejas o grupos?',
 '¿Están los/las estudiantes realizando las actividades por sí mismos, sin ayuda del docente?',
 'Agregue cualquier comentario adicional, que sea relevante para entender lo que está sucediendo en el aula en este instante de la clase']
instantaneas = instantaneas.reindex(columns=nuevo_orden)

In [14]:
instantaneas

,ID de respuesta,Número de instantánea,Visita,Número de documento de docente observado/a,Nombre de docente observado/a,Indique sexo del docente,¿El/la docente trabaja con una guía pedagógica?,Información de la clase Asignatura - Selected Choice,Código IE,¿Qué está haciendo el/la docente ahora?,...,¿Quiénes están socializando su trabajo?,"¿Están los/las estudiantes distraídos, haciendo indisciplina o mostrando de alguna otra manera que no están involucrados en las actividades que lidera el/la docente?",¿Quiénes no están involucrados en las actividades que lidera el/la docente?,¿Están haciendo uso de herramientas computacionales?,¿Quiénes están haciendo uso de herramientas computacionales?,¿Están los/las estudiantes trabajando individualmente?,¿Están los/las estudiantes trabajando en parejas o grupos?,¿Quiénes están ejerciendo roles de liderazgo en el trabajo en parejas o grupos?,"¿Están los/las estudiantes realizando las actividades por sí mismos, sin ayuda del docente?","Agregue cualquier comentario adicional, que sea relevante para entender lo que está sucediendo en el aula en este instante de la clase"
2,R_7esDIa291bLHEUw,1,2,3.379498e+06,Fabio Restrepo Restrepo,Hombre,Sí,Tecnología e Informática,NPC064,Está explicando conceptos de pensamiento compu...,...,NaN,Ninguno,NaN,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Todos(as),Ninguno,NaN
1,R_7r8EwEBEf2yeEhw,1,3,3.379498e+06,Fabio Restrepo Restrepo,Hombre,No,Tecnología e Informática,NPC064,Está explicando conceptos de pensamiento compu...,...,NaN,Uno o dos,Sólo los niños,Ninguno. No se están realizando actividades de...,Todos(as),Todos(as),Niños y niñas por igual,Todos(as),Ninguno,NaN
9,R_1ffrXqDCHFeK9vi,1,1,4.238721e+06,Luis Raúl Barón Manrique,Hombre,Sí,Tecnología e Informática,NPC110,Está explicando conceptos de pensamiento compu...,...,NaN,Uno o dos,Sólo los niños,Ninguno. No se están realizando actividades de...,Todos(as),Todos(as),Niños y niñas por igual,Todos(as),Ninguno,NaN
8,R_3SjC2EpHxhXK6yk,1,3,4.238721e+06,Luis Raúl Barón,Hombre,Sí,Tecnología e Informática,NPC110,Está explicando conceptos de pensamiento compu...,...,NaN,Ninguno,NaN,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Ninguno,Ninguno,NaN
11,R_5qjsHomeq9XN5F7,1,1,4.238845e+06,Hernando Diaz,Hombre,No,Tecnología e Informática,NPC234,Está haciendo preguntas a toda la clase,...,NaN,Pocos,Niños y niñas por igual,Ninguno. No se están realizando actividades de...,La mayoría,La mayoría,Niños y niñas por igual,Ninguno,Todos(as),Niños y niñas por igual
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,R_1ahGCVUVPVaxvGN,12,3,1.125763e+09,Yesenia Figueredo,Mujer,Sí,Tecnología e Informática,NPC071,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
800,R_7nIaUt1Pf0Z2r9l,12,1,1.127344e+09,José René Bernal Piza,Hombre,Sí,Tecnología e Informática,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
799,R_1efRKcyfddxQ0y0,12,3,1.127344e+09,José René Bernal Piza,Hombre,Sí,Tecnología e Informática,NPC324,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
806,R_7j6uWowM7Mi3LNj,12,1,1.130621e+09,Aleida Mabel Rosero Hernández,Mujer,Sí,Otro:,NPC222,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
rename_dict = {
    'Fecha de inicio': 'fecha_inicio',
    'Fecha de finalización': 'fecha_fin',
    'Visita': 'visita',
    'Momento': 'momento',
    'Tipo de respuesta': 'tipo_respuesta',
    'Dirección IP': 'ip',
    'Progreso': 'progreso',
    'Duración (en segundos)': 'duracion_seg',
    'Finalizado': 'finalizado',
    'Fecha registrada': 'fecha_registro',
    'ID de respuesta': 'id_respuesta',
    'Apellido del destinatario': 'apellido_dest',
    'Nombre del destinatario': 'nombre_dest',
    'Correo electrónico del destinatario': 'correo_dest',
    'Referencia a datos externos': 'ref_externa',
    'Latitud de la ubicación': 'latitud',
    'Longitud de la ubicación': 'longitud',
    'Canal de la distribución': 'canal_dist',
    'Idioma del usuario': 'idioma',
    'Fecha del acompañamiento Indique la fecha en que realizó el acompañamiento en aula. Use el formato dd/mm/aaaa': 'fecha_acomp',
    'Hora de inicio de la clase (Formato Hora Militar) Indique la hora en la que inició la clase acompañada.': 'hora_inicio',
    'Unidad': 'unidad',
    'Mentor': 'mentor',
    'UG1': 'ug_1',
    'Número de documento de docente observado/a': 'doc_docente',
    'Nombre de docente observado/a': 'nombre_docente',
    'Indique sexo del docente': 'sexo_docente',
    'Información de la clase Asignatura - Selected Choice': 'asignatura',
    'Información de la clase Asignatura - Otro: - Texto': 'otra_asignatura',
    'Grado': 'grado',
    'Al cierre de la clase ¿Se hace uso de algún tipo de gráfico de anclaje o memoria colectiva?': 'uso_grafico_anclaje',
    '¿El/la docente trabaja con una guía pedagógica?': 'guia_pedagogica',
    '¿A qué grado corresponde la guía pedagógica empleada?': 'grado_guia',
    '¿Con que número de guía se trabaja?': 'num_guia',
    '¿Con qué sesión de la guía se trabaja?': 'sesion_guia',
    'Tema de la clase Si no se trabaja con una guía pedagógica, por favor indique el tema de la clase': 'tema_clase',
    'Número de estudiantes en total Número entero sin puntos': 'total_estudiantes',
    'Número de estudiantes de sexo femenino Número entero sin puntos': 'est_femenino',
    'Número de estudiantes de sexo masculino Número entero sin puntos': 'est_masculino',
    'Duración estimada de la clase (horas y minutos) Por favor, escriba el tiempo estimado que debe durar la clase observada en formato horas y minutos (00:00). Por ejemplo, si la clase dura 50 minutos, debe colocar 00:50.': 'duracion_clase',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Selected Choice - Computadores portátiles': 'tech_portatiles',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Selected Choice - Computadores de escritorio': 'tech_escritorio',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Selected Choice - Tabletas': 'tech_tabletas',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Selected Choice - Celulares': 'tech_celulares',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Selected Choice - Microprocesadores (tarjetas arduino, micro:bit, raspberry pi o semejantes)': 'tech_microprocesadores',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Selected Choice - Componentes electrónicos externos (sensores, servos, LEDS, etc)': 'tech_componentes_externos',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Selected Choice - Robots': 'tech_robots',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Selected Choice - No se hace uso de tecnologías digitales': 'tech_sin_uso',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Selected Choice - Otro:': 'tech_otra',
    'Tecnologías digitales disponibles en la clase Elija todas las que apliquen - Otro: - Texto': 'tech_otra_detalle',
    'Timing - Primer clic': 'click_inicio',
    'Timing - Último clic': 'click_fin',
    'Timing - Envío de página': 'envio_pagina',
    'Timing - Recuento de clics': 'num_clics',
    'Al inicio de la clase ¿Cuántos minutos transcurren entre la hora de inicio y el momento en que el docente empieza a gestionar el aula para empezar la clase? Escriba la cantidad de minutos en números enteros. Nota: Las actividades gestión inicial como llamar a lista o seguir rutinas de inicio establecidas, se consideran parte de la clase.': 'min_inicio_gestion',
    '¿Se presentan los objetivos de aprendizaje de la lección?': 'objetivos_aprend',
    '¿Se exploran los conocimientos previos de los estudiantes y su conexión con los temas de la lección?': 'conoc_previos',
    '¿Se presentan claramente los conceptos claves que serán usados en la lección? Nota: Las revisiones de conceptos presentados en lecciones anteriores cuentan como parte de la presentación inicial': 'conceptos_clave',
    '¿Se pide a los/las estudiantes realizar alguna actividad desconectada? Marque sí si en la clase se realiza alguna actividad desconectada para promover el pensamiento computacional. Ej. Se implementa alguna actividad de Code.org, las fichas CfK, las guías pedagógicas, Bebras, etc. que no requiera el uso de dispositivos electrónicos.': 'act_desconectada_presente',
    'Describa brevemente la actividad desconectada que se realiza durante la clase Si la actividad fue tomada de algún recurso/fuente que usted pueda identificar, por favor menciónelo. Ej. Actividad desconectada ficha 1 Cfk - desplazamiento de elementos a través de un tablero con obstáculos': 'act_desconectada_desc',
    '¿Cuántos estudiantes realizan la actividad desconectada de la manera esperada?': 'act_desconectada_participantes',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Selected Choice - Aparente dificultad con la comprensión de las instrucciones': 'razon_dificultad_instrucciones',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Selected Choice - Aparente dificultad con la comprensión del concepto': 'razon_dificultad_concepto',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Selected Choice - Aparente dificultad con la comprensión de la narrativa (contexto) de la actividad': 'razon_dificultad_narrativa',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Selected Choice - Otro:': 'razon_dificultad_otra',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Otro: - Texto': 'razon_dificultad_otra_detalle',
    '¿Se invita a las/los estudiantes a compartir la solución que dieron a la actividad desconectada y reflexionar sobre otras posibles soluciones?': 'act_desconectada_compartir',
    'Onda semántica: ¿Se hace un cierre formal de la actividad desconectada para destacar la relación entre ésta y el tema o concepto presentado?': 'act_desconectada_cierre',
    '¿La actividad desconectada que se propuso permitió efectivamente aplicar los conceptos y/o subhabilidades asociados al tema central de la clase?': 'act_desconectada_eficacia',
    'Escriba cualquier comentario adicional que tenga y que permita aclarar mejor su respuesta a la pregunta anterior.': 'act_desconectada_comentarios',
    'Preguntas para clases conectadas ¿Se realizan actividades que requieren la programación y/o el uso de algún dispositivo electrónico? Marque sí, si en la clase se hace uso de algún computador, tableta, micro:procesador, robot y/o elemento electrónico que forma parte de un kit de computación física': 'act_conectada_presente',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se cuenta con los dispositivos de cómputo funcionales necesarios para que todos los/las estudiantes puedan trabajar sólos o en parejas': 'act_conectada_dispositivos',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se cuenta con los microprocesadores y accesorios necesarios': 'act_conectada_microprocesadores',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se distribuyen/asignan de forma ordenada los materiales y recursos de hardware requeridos': 'act_conectada_distribucion',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se cuenta con acceso a los programas (software) que se requiere': 'act_conectada_software',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se utilizan herramientas tecnológicas adicionales para apoyar el desarrollo de la sesión': 'act_conectada_herramientas',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se cuenta con la conectividad requerida para el desarrollo de las actividades propuestas': 'act_conectada_conectividad',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Predecir: El/La docente presenta un código y pregunta a sus estudiantes lo que creen que este hará': 'act_conectada_predecir',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Ejecutar/Usa: El/La docente pide a sus estudiantes que repliquen el código que presentó y comprueben lo que este hace': 'act_conectada_ejecutar_replicar',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Ejecutar/Usa: El/La docente entrega a sus estudiantes el código y les pide que lo ejecuten para comprobar lo que hace': 'act_conectada_ejecutar_entregar',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Investigar: Los/las estudiantes exploran libremente líneas o bloques de código, sin ningún tipo de orientación o guía del docente': 'act_conectada_investigar_libre',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Investigar: El/La docente pide a sus estudiantes que exploren ciertos comandos o bloques del lenguaje de programación, y les da preguntas específicas que deben responder al respecto de estos': 'act_conectada_investigar_guiada',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Investigar: El/La docente pide a sus estudiantes que compartan sus hallazgos/aprendizajes/descubrimientos con relación a los bloques explorados y/o los usados en el código inicialmente presentado': 'act_conectada_investigar_compartir',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - El/La docente explica, amplía y/o aclara información sobre los comandos o bloques de programación que se utilizan en el código usado': 'act_conectada_explicacion',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Modificar/Modifica: Los/las estudiantes hacen cambios al código modelo sin ayuda del docente': 'act_conectada_modificar_ind',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Modificar/Modifica: Los/las estudiantes hacen cambios al código modelo, pero es el/la docente quien lidera estos cambios': 'act_conectada_modificar_docente',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Modificar/Modifica: Los/las estudiantes hacen cambios al código modelo con apoyo ocasional del docente': 'act_conectada_modificar_apoyo',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Hacer/Crea: Los/las estudiantes resuelven por sí mismos un reto de programación que les requiere hacer uso de lo aprendido en las etapas previas de la clase': 'act_conectada_hacer_ind',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Hacer/Crea: Los/las estudiantes resuelven, con apoyo ocasional del docente, un reto de programación que les requiere hacer uso de lo aprendido en las etapas previas de la clase': 'act_conectada_hacer_apoyo',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Hacer/Crea: Los/las estudiantes replican la solución a un reto de programación que resuelve el/la docente': 'act_conectada_hacer_replicar',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Los/las estudiantes trabajan en la creación de un proyecto de computación física con el apoyo ocasional del docente': 'act_conectada_proyecto_fisico',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Programación por pares: Los/las estudiantes resuelven los retos de programación propuestos, trabajando en parejas y alternandose el uso del computador': 'estrategia_pares',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Preguntas de Parsons: El/La docente muestra líneas de código o bloques desordenados y pide a sus estudiantes que las analicen y reorganicen, justificando luego sus decisiones': 'estrategia_parsons',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Programación en vivo: El/La docente demuesta la creación de un programa y va explicando en voz alta el proceso y las razones por las que elige cada línea de código o bloque y no otros': 'estrategia_vivo',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Lectura de código: El/La docente muestra un código y pide a sus estudiantes que respondan preguntas y/o expliquen el objetivo de este y de las líneas o bloques que lo componen': 'estrategia_lectura',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Evaluación de pares: El/La docente pide a sus estudiantes que trabajen en parejas, intercambien sus programas y se evalúen uno a otro siguiendo algunos criterios': 'estrategia_evaluacion',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Aprendizaje Basado en Proyectos: Los/las estudiantes avanzan el desarrollo de un proyecto de computación física': 'estrategia_proyectos',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Pensamiento de Diseño: Los/las estudiantes avanzan el desarrollo de un proyecto de computación física, enmarcado en al menos una de las 5 etapas del Pensamiento de Diseño: empatizar, definir, idear, prototipar y evaluar': 'estrategia_diseno',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Tinkering: Los/las estudiantes exploran libremente el ensamblaje de componentes electrónicos, ideando soluciones, sin intervención del docente': 'estrategia_tinkering',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - No se observa ninguna de las anteriores estrategias': 'estrategia_ninguna',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Otro:': 'estrategia_otra',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Otro: - Texto': 'estrategia_otra_detalle',
    '¿Cuántos estudiantes logran solucionar el reto de codificación propuesto por el/la docente?': 'estudiantes_solucion',
    '¿Se promueve la idea de que todos y todas pueden desarrollar su pensamiento computacional y habilidades de programación?': 'promocion_comp',
    'Por favor, describa brevemente las actividades conectadas que se efectúan en la clase observada y agregue cualquier otro comentario que permita entender sus respuestas a las preguntas anteriores': 'act_conectada_comentarios',
    '¿Se usa el vocabulario adecuado (terminología correcta) con relación al pensamiento computacional y/o las habilidades de programación?': 'vocabulario_comp',
    '¿Se conectan los temas presentados con la vida diaria?': 'conexion_vida',
    '¿Se promueven los procesos de metacognición y reflexión?': 'metacognicion',
    '¿Sabe cómo resolver los problemas técnicos cuando fallan las herramientas computacionales en el aula?': 'resolucion_tecnica',
    'Si aplica, por favor, indique las dificultades técnicas que se presentaron durante la clase acompañada': 'dificultades_tecnicas',
    'Sobre la gestión de aula ¿Se preparó de forma previa el material requerido para la lección?': 'prep_material',
    '¿Se gestionan correctamente los materiales e instrumentos para el desarrollo de las actividades propuestas?': 'gestion_material',
    '¿Se valora el esfuerzo de los estudiantes para desarrollar las actividades propuestas?': 'valoracion_esfuerzo',
    '¿Se observa acompañamiento con estrategias de apoyo (aclaración de dudas, explicaciones, ejemplos adicionales, invitación a revisar notas de clases previas, etc) a los estudiantes durante el desarrollo de las actividades?': 'apoyo_estudiantes',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Se corrigen comentarios y comportamientos sexistas': 'equidad_corrige_sexismo',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Se estimula el liderazgo femenino': 'equidad_liderazgo_fem',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Se realizan acciones afirmativas en términos de género': 'equidad_acciones_afirm',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Se dedica tiempo de la clase a hacer reflexiones sobre equidad de género, por ejemplo, destacando los aportes de personajes masculinos y femeninos': 'equidad_reflexion',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Se hace uso de lenguaje inclusivo, sin estereotipos de género, que promueva respeto por la diversidad': 'equidad_lenguaje_inc',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - No se observa ninguna práctica pedagógica en pro de la equidad de género': 'equidad_ninguna',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Otra': 'equidad_otra',
    'Otra.2': 'equidad_otra_detalle',
    'Por favor, agregue cualquier otra información adicional que considere relevante para aclarar sus respuestas a la pregunta anterior': 'info_adicional_equidad',
    'Instantánea 1 - minuto 8 de la observación ¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante.': 'accion_docente',
    'Indique con quiénes está interactuando el/la docente': 'interaccion',
    '¿Con quiénes está interactuando el/la docente?': 'interaccion_det',
    '¿Están los/las estudiantes respondiendo las preguntas y/o participando en las discusiones?': 'participacion',
    '¿Están los/las estudiantes escuchando atentamente al docente?': 'escucha',
    '¿Están los/las estudiantes tomando nota de las explicaciones o discusiones?': 'notas',
    '¿Están los/las estudiantes haciendo preguntas y/o pidiendo ayuda al docente?': 'preguntas',
    '¿Están los/las estudiantes socializando su trabajo?': 'socializacion',
    '¿Quiénes están socializando su trabajo?': 'socializacion_quien',
    '¿Están los/las estudiantes distraídos, haciendo indisciplina o mostrando de alguna otra manera que no están involucrados en las actividades que lidera el/la docente?': 'distraccion',
    '¿Quiénes no están involucrados en las actividades que lidera el/la docente?': 'no_involucrados',
    '¿Están haciendo uso de herramientas computacionales?': 'uso_tech',
    '¿Quiénes están haciendo uso de herramientas computacionales?': 'uso_tech_quien',
    '¿Están los/las estudiantes trabajando individualmente?': 'trabajo_ind',
    '¿Están los/las estudiantes trabajando en parejas o grupos?': 'trabajo_grupo',
    '¿Quiénes están ejerciendo roles de liderazgo en el trabajo en parejas o grupos?': 'liderazgo',
    '¿Están los/las estudiantes realizando las actividades por sí mismos, sin ayuda del docente?': 'autonomia',
    'Agregue cualquier comentario adicional, que sea relevante para entender lo que está sucediendo en el aula en este instante de la clase': 'comentarios',
    '¿Avanzar a la Instantánea 2?': 'avanzar',
    'Por favor, seleccione qué desea hacer.': 'accion_final',
    'Al inicio de la clase ¿Cuántos minutos transcurren entre el inicio de la clase estipulada y el inicio de la clase real? Escriba la cantidad de minutos en números enteros. Nota: las actividades de gestión inicial como llamar a la lista o seguir rutinas de inicio establecidas se consideran parte de la clase.': 'min_inicio_gestion',
    '¿Se pide a los/las estudiantes realizar alguna actividad desconectada? Marque sí si en la clase se realiza alguna actividad desconectada para promover el pensamiento computacional. Ej. Se implementa alguna actividad de Code.org, las fichas CfK, las guías pedagógicas, Bebras, etc. que no requiera el uso de dispositivos electrónicos.': 'act_desconectada_presente',
    'Describa brevemente la actividad desconectada que se realiza durante la clase Si la actividad fue tomada de algún recurso/fuente que usted pueda identificar, por favor menciónelo. Ej. Actividad desconectada ficha 1 Cfk - desplazamiento de elementos a través de un tablero con obstáculos': 'act_desconectada_desc',
    '¿Cuántos estudiantes realizan la actividad desconectada de la manera esperada?': 'act_desconectada_participantes',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Selected Choice - Aparente dificultad con la comprensión de las instrucciones': 'razon_dificultad_instrucciones',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Selected Choice - Aparente dificultad con la comprensión del concepto': 'razon_dificultad_concepto',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Selected Choice - Aparente dificultad con la comprensión de la narrativa (contexto) de la actividad': 'razon_dificultad_narrativa',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Selected Choice - Otro:': 'razon_dificultad_otra',
    'Por favor, indique las razones por las que, a su juicio, hay estudiantes que no logran desarrollar la actividad desconectada según lo esperado Elija todas las que apliquen - Otro: - Texto': 'razon_dificultad_otra_detalle',
    '¿Se invita a las/los estudiantes a compartir la solución que dieron a la actividad desconectada y reflexionar sobre otras posibles soluciones?': 'act_desconectada_compartir',
    'Onda semántica: ¿Se hace un cierre formal de la actividad desconectada para destacar la relación entre ésta y el tema o concepto presentado?': 'act_desconectada_cierre',
    '¿La actividad desconectada que se propuso permitió efectivamente aplicar los conceptos y/o subhabilidades asociados al tema central de la clase?': 'act_desconectada_eficacia',
    'Escriba cualquier comentario adicional que tenga y que permita aclarar mejor su respuesta a la pregunta anterior.': 'act_desconectada_comentarios',
    'Preguntas para clases conectadas ¿Se realizan actividades que requieren la programación y/o el uso de algún dispositivo electrónico? Marque sí, si en la clase se hace uso de algún computador, tableta, micro:procesador, robot y/o elemento electrónico que forma parte de un kit de computación física': 'act_conectada_presente',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se cuenta con los dispositivos de cómputo funcionales necesarios para que todos los/las estudiantes puedan trabajar sólos o en parejas': 'act_conectada_dispositivos',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se cuenta con los microprocesadores y accesorios necesarios': 'act_conectada_microprocesadores',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se distribuyen/asignan de forma ordenada los materiales y recursos de hardware requeridos': 'act_conectada_distribucion',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se cuenta con acceso a los programas (software) que se requiere': 'act_conectada_software',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se utilizan herramientas tecnológicas adicionales para apoyar el desarrollo de la sesión': 'act_conectada_herramientas',
    'Elija todas las afirmaciones que representen lo que ocurre en la clase - Se cuenta con la conectividad requerida para el desarrollo de las actividades propuestas': 'act_conectada_conectividad',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Predecir: El/La docente presenta un código y pregunta a sus estudiantes lo que creen que este hará': 'act_conectada_predecir',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Ejecutar/Usa: El/La docente pide a sus estudiantes que repliquen el código que presentó y comprueben lo que este hace': 'act_conectada_ejecutar_replicar',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Ejecutar/Usa: El/La docente entrega a sus estudiantes el código y les pide que lo ejecuten para comprobar lo que hace': 'act_conectada_ejecutar_entregar',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Investigar: Los/las estudiantes exploran libremente líneas o bloques de código, sin ningún tipo de orientación o guía del docente': 'act_conectada_investigar_libre',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Investigar: El/La docente pide a sus estudiantes que exploren ciertos comandos o bloques del lenguaje de programación, y les da preguntas específicas que deben responder al respecto de estos': 'act_conectada_investigar_guiada',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Investigar: El/La docente pide a sus estudiantes que compartan sus hallazgos/aprendizajes/descubrimientos con relación a los bloques explorados y/o los usados en el código inicialmente presentado': 'act_conectada_investigar_compartir',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - El/La docente explica, amplía y/o aclara información sobre los comandos o bloques de programación que se utilizan en el código usado': 'act_conectada_explicacion',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Modificar/Modifica: Los/las estudiantes hacen cambios al código modelo sin ayuda del docente': 'act_conectada_modificar_ind',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Modificar/Modifica: Los/las estudiantes hacen cambios al código modelo, pero es el/la docente quien lidera estos cambios': 'act_conectada_modificar_docente',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Modificar/Modifica: Los/las estudiantes hacen cambios al código modelo con apoyo ocasional del docente': 'act_conectada_modificar_apoyo',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Hacer/Crea: Los/las estudiantes resuelven por sí mismos un reto de programación que les requiere hacer uso de lo aprendido en las etapas previas de la clase': 'act_conectada_hacer_ind',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Hacer/Crea: Los/las estudiantes resuelven, con apoyo ocasional del docente, un reto de programación que les requiere hacer uso de lo aprendido en las etapas previas de la clase': 'act_conectada_hacer_apoyo',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Hacer/Crea: Los/las estudiantes replican la solución a un reto de programación que resuelve el/la docente': 'act_conectada_hacer_replicar',
    'Seleccione todas las opciones que mejor describan lo que sucede durante el desarrollo de la actividad conectada - Los/las estudiantes trabajan en la creación de un proyecto de computación física con el apoyo ocasional del docente': 'act_conectada_proyecto_fisico',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Programación por pares: Los/las estudiantes resuelven los retos de programación propuestos, trabajando en parejas y alternandose el uso del computador': 'estrategia_pares',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Preguntas de Parsons: El/La docente muestra líneas de código o bloques desordenados y pide a sus estudiantes que las analicen y reorganicen, justificando luego sus decisiones': 'estrategia_parsons',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Programación en vivo: El/La docente demuesta la creación de un programa y va explicando en voz alta el proceso y las razones por las que elige cada línea de código o bloque y no otros': 'estrategia_vivo',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Lectura de código: El/La docente muestra un código y pide a sus estudiantes que respondan preguntas y/o expliquen el objetivo de este y de las líneas o bloques que lo componen': 'estrategia_lectura',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Evaluación de pares: El/La docente pide a sus estudiantes que trabajen en parejas, intercambien sus programas y se evalúen uno a otro siguiendo algunos criterios': 'estrategia_evaluacion',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Aprendizaje Basado en Proyectos: Los/las estudiantes avanzan el desarrollo de un proyecto de computación física': 'estrategia_proyectos',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Pensamiento de Diseño: Los/las estudiantes avanzan el desarrollo de un proyecto de computación física, enmarcado en al menos una de las 5 etapas del Pensamiento de Diseño: empatizar, definir, idear, prototipar y evaluar': 'estrategia_diseno',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Tinkering: Los/las estudiantes exploran libremente el ensamblaje de componentes electrónicos, ideando soluciones, sin intervención del docente': 'estrategia_tinkering',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - No se observa ninguna de las anteriores estrategias': 'estrategia_ninguna',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Selected Choice - Otro:': 'estrategia_otra',
    'Estrategias pedagógicas y didácticas: Elija todas las que observe durante la clase - Otro: - Texto': 'estrategia_otra_detalle',
    '¿Cuántos estudiantes logran solucionar el reto de codificación propuesto por el/la docente?': 'estudiantes_solucion',
    '¿Se promueve la idea de que todos y todas pueden desarrollar su pensamiento computacional y habilidades de programación?': 'promocion_comp',
    'Por favor, describa brevemente las actividades conectadas que se efectúan en la clase observada y agregue cualquier otro comentario que permita entender sus respuestas a las preguntas anteriores': 'act_conectada_comentarios',
    '¿Cuántos estudiantes logran seguir las instrucciones de las actividades propuestas, sin necesidad de aclaraciones posteriores?': 'estudiantes_instrucciones',
    'Por favor, indique las razones por las que todos(as) o la mayoría de estudiantes logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Selected Choice - El/La docente se asegura de tener la atención de todo el grupo antes de dar instrucciones': 'razon_inicio_atencion',
    'Por favor, indique las razones por las que todos(as) o la mayoría de estudiantes logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Selected Choice - El/La docente provee instrucciones claras': 'razon_inicio_claras',
    'Por favor, indique las razones por las que todos(as) o la mayoría de estudiantes logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Selected Choice - El/La docente modela (ejemplifica) la actividad que se va a desarrollar': 'razon_inicio_modelo_doc',
    'Por favor, indique las razones por las que todos(as) o la mayoría de estudiantes logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Selected Choice - El/La docente pide al menos a un(a) estudiante que modele (ejemplifique) la actividad': 'razon_inicio_modelo_est',
    'Por favor, indique las razones por las que todos(as) o la mayoría de estudiantes logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Selected Choice - Otro:': 'razon_inicio_otra',
    'Por favor, indique las razones por las que todos(as) o la mayoría de estudiantes logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Otro: - Texto': 'razon_inicio_otra_detalle',
    'Por favor, indique las razones por las que los/las estudiantes no logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Selected Choice - El/La docente da instrucciones confusas': 'razon_no_inicio_confusas',
    'Por favor, indique las razones por las que los/las estudiantes no logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Selected Choice - El/La docente pide que se inicie la actividad, pero no da instrucciones sobre lo que se debe realizar': 'razon_no_inicio_sin_instrucciones',
    'Por favor, indique las razones por las que los/las estudiantes no logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Selected Choice - El/La docente pide que se inicie la actividad sin haberla modelado previamente': 'razon_no_inicio_sin_modelo',
    'Por favor, indique las razones por las que los/las estudiantes no logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Selected Choice - Otro:': 'razon_no_inicio_otra',
    'Por favor, indique las razones por las que los/las estudiantes no logran empezar el desarrollo de las actividades después de las instrucciones del docente Elija todas las que apliquen - Otro: - Texto': 'razon_no_inicio_otra_detalle',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Selected Choice - Se corrigen comentarios y comportamientos sexistas': 'equidad_corrige_sexismo',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Selected Choice - Se estimula el liderazgo femenino': 'equidad_liderazgo_fem',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Selected Choice - Se realizan acciones afirmativas en términos de género': 'equidad_acciones_afirm',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Selected Choice - Se dedica tiempo de la clase a hacer reflexiones sobre equidad de género, por ejemplo, destacando los aportes de personajes masculinos y femeninos': 'equidad_reflexion',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Selected Choice - No se observa ninguna práctica pedagógica en pro de la equidad': 'equidad_ninguna',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Selected Choice - Otro:': 'equidad_otra',
    'Prácticas en pro de la equidad de género Elija todas las opciones que representen lo que haya sucedido durante la clase acompañada. - Otro: - Texto': 'equidad_otra_detalle'
}

In [16]:
def rename_columns(df, rename_dict):
    """
    Renombra las columnas de un DataFrame usando un diccionario.
    - Ignora claves en rename_dict que no coincidan con columnas existentes.
    - Mantiene los nombres originales si no hay nuevo nombre en rename_dict.

    Args:
        df (pandas.DataFrame): DataFrame cuyas columnas se van a renombrar.
        rename_dict (dict): Diccionario con {nombre_actual: nuevo_nombre}.

    Returns:
        pandas.DataFrame: DataFrame con las columnas renombradas.
    """
    # Crea una copia del DataFrame para evitar modificar el original
    df_renamed = df.copy()

    # Obtiene las columnas actuales del DataFrame
    current_columns = df_renamed.columns.tolist()


    # Solo incluye pares donde la clave está en las columnas actuales
    safe_rename_dict = {col: rename_dict[col] for col in current_columns if col in rename_dict}

    # Aplica el renombramiento
    df_renamed = df_renamed.rename(columns=safe_rename_dict)

    return df_renamed

In [17]:
# Obtiene todas las columnas hasta la columna de interés
columnas_hasta_instantanea = df.columns[:df.columns.get_loc('Instantánea 1 - minuto 8 de la observación ¿Qué está haciendo el/la docente ahora? Indique cuál de las siguientes acciones es la principal que está realizando la/el docente en este instante.') + 1]

# Asegúrate de que 'Código IE' no esté ya incluida si está en ese rango
if 'Código IE' not in columnas_hasta_instantanea:
    # Crea la lista final de columnas a seleccionar
    columnas_a_seleccionar = columnas_hasta_instantanea.tolist() + ['Código IE']
else:
    columnas_a_seleccionar = columnas_hasta_instantanea.tolist() 

columnas_a_seleccionar = columnas_a_seleccionar + ['Visita', 'Momento', '¿Se promueven los procesos de metacognición y reflexión?', 'Al cierre de la clase ¿Se hace uso de algún tipo de gráfico de anclaje o memoria colectiva?']

# Selecciona las columnas para crear Obs_generales
obs_generales = df.loc[:, columnas_a_seleccionar].copy()

In [18]:
# Llama a la función
instantaneas = rename_columns(instantaneas, rename_dict)
obs_generales = rename_columns(obs_generales, rename_dict)

In [19]:
[c for c in obs_generales.columns if 'metacognicion' in c.lower()]

['metacognicion']

In [20]:
instantaneas["¿Qué está haciendo el/la docente ahora?"].unique()

array(['Está explicando conceptos de pensamiento computacional y/o programación',
       'Está haciendo preguntas a toda la clase',
       'Está dando instrucciones para el desarrollo de una actividad',
       'Está monitoreando o supervisando el trabajo de los/las estudiantes, sin intervenir',
       'Está copiando en el tablero sin hablar',
       'Está escuchando las intervenciones o presentaciones de los/las estudiantes',
       'Está retroalimentando el trabajo realizado por los/las estudiantes (en parejas o grupos)',
       'Está resolviendo problemas técnicos (uso del proyector, problemas con el computador, etc)',
       'Está haciendo actividades de gestión de aula (disciplina, atención de los/las estudiantes, etc)',
       'Está explicando conceptos relevantes para la lección, pero no asociados al pensamiento computacional o la programación',
       'Está presente físicamente, pero desconectado(a) de las actividades',
       'Está hablando de temas no relacionados con la clase

In [21]:
instantaneas = instantaneas.replace(['nan', 'NaN', 'N/A'], '', regex=False)
obs_generales = obs_generales.replace(['nan', 'NaN', 'N/A'], '', regex=False)

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_16612\4285620766.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  instantaneas = instantaneas.replace(['nan', 'NaN', 'N/A'], '', regex=False)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_16612\4285620766.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  obs_generales = obs_generales.replace(['nan', 'NaN', 'N/A'], '', regex=False)


In [22]:
nuevo_orden = [
    # Metadatos generales y fechas
    'fecha_inicio',
    'fecha_fin',
    'visita',
    'momento',
    'fecha_acomp',
    'fecha_registro',
    'hora_inicio',
    'duracion_clase',
    'duracion_seg',

    # Identificación y datos del docente
    'doc_docente',
    'nombre_docente',
    'sexo_docente',
    'unidad',
    'Código',

    # Respuesta y progreso
    'tipo_respuesta',
    'id_respuesta',
    'ip',
    'progreso',
    'finalizado',

    # Ubicación y distribución
    'latitud',
    'longitud',
    'canal_dist',
    'idioma',

    # Detalles de la clase
    'asignatura',
    'otra_asignatura',
    'grado',
    'guia_pedagogica',
    'grado_guia',
    'num_guia',
    'sesion_guia',
    'tema_clase',
    'total_estudiantes',
    'est_femenino',
    'est_masculino',

    # Tecnologías digitales
    'tech_portatiles',
    'tech_escritorio',
    'tech_tabletas',
    'tech_celulares',
    'tech_microprocesadores',
    'tech_componentes_externos',
    'tech_robots',
    'tech_sin_uso',
    'tech_otra',
    'tech_otra_detalle',

    # Interacciones y timing
    'click_inicio',
    'click_fin',
    'envio_pagina',
    'num_clics',
    'min_inicio_gestion',
    'objetivos_aprend',
    'conoc_previos',
    'conceptos_clave',

    # Actividades desconectadas
    'act_desconectada_presente',
    'act_desconectada_desc',
    'act_desconectada_participantes',
    'razon_dificultad_instrucciones',
    'razon_dificultad_concepto',
    'razon_dificultad_narrativa',
    'razon_dificultad_otra',
    'razon_dificultad_otra_detalle',
    'act_desconectada_compartir',
    'act_desconectada_cierre',
    'act_desconectada_eficacia',
    'act_desconectada_comentarios',

    # Actividades conectadas
    'act_conectada_presente',
    'act_conectada_dispositivos',
    'act_conectada_microprocesadores',
    'act_conectada_distribucion',
    'act_conectada_software',
    'act_conectada_herramientas',
    'act_conectada_conectividad',
    'act_conectada_predecir',
    'act_conectada_ejecutar_replicar',
    'act_conectada_ejecutar_entregar',
    'act_conectada_investigar_libre',
    'act_conectada_investigar_guiada',
    'act_conectada_investigar_compartir',
    'act_conectada_explicacion',
    'act_conectada_modificar_ind',
    'act_conectada_modificar_docente',
    'act_conectada_modificar_apoyo',
    'act_conectada_hacer_ind',
    'act_conectada_hacer_apoyo',
    'act_conectada_hacer_replicar',
    'act_conectada_proyecto_fisico',
    'act_conectada_comentarios',

    # Estrategias pedagógicas
    'estrategia_pares',
    'estrategia_parsons',
    'estrategia_vivo',
    'estrategia_lectura',
    'estrategia_evaluacion',
    'estrategia_proyectos',
    'estrategia_diseno',
    'estrategia_tinkering',
    'estrategia_ninguna',
    'estrategia_otra',
    'estrategia_otra_detalle',

    # Evaluación de estudiantes
    'estudiantes_solucion',
    'estudiantes_instrucciones',
    'razon_inicio_atencion',
    'razon_inicio_claras',
    'razon_inicio_modelo_doc',
    'razon_inicio_modelo_est',
    'razon_inicio_otra',
    'razon_inicio_otra_detalle',
    'razon_no_inicio_confusas',
    'razon_no_inicio_sin_instrucciones',
    'razon_no_inicio_sin_modelo',
    'razon_no_inicio_otra',
    'razon_no_inicio_otra_detalle',

    # Promoción y competencias
    'promocion_comp',
    'vocabulario_comp',
    'conexion_vida',
    'resolucion_tecnica',
    'dificultades_tecnicas',

    # Gestión y equidad
    'prep_material',
    'gestion_material',
    'valoracion_esfuerzo',
    'apoyo_estudiantes',
    'equidad_corrige_sexismo',
    'equidad_liderazgo_fem',
    'equidad_acciones_afirm',
    'equidad_reflexion',
    'equidad_ninguna',
    'equidad_otra',
    'equidad_otra_detalle',
    'info_adicional_equidad',
    'metacognicion',
    'uso_grafico_anclaje'

]

In [23]:
obs_generales = obs_generales.reindex(columns=nuevo_orden)

In [24]:
['Está explicando conceptos de pensamiento computacional y/o programación',
       'Está haciendo preguntas a toda la clase',
       'Está dando instrucciones para el desarrollo de una actividad',
       'Está monitoreando o supervisando el trabajo de los/las estudiantes, sin intervenir',
       'Está copiando en el tablero sin hablar',
       'Está escuchando las intervenciones o presentaciones de los/las estudiantes',
       'Está retroalimentando el trabajo realizado por los/las estudiantes (en parejas o grupos)',
       'Está resolviendo problemas técnicos (uso del proyector, problemas con el computador, etc)',
       'Está haciendo actividades de gestión de aula (disciplina, atención de los/las estudiantes, etc)',
       'Está explicando conceptos relevantes para la lección, pero no asociados al pensamiento computacional o la programación',
       'Está presente físicamente, pero desconectado(a) de las actividades',
       'Está hablando de temas no relacionados con la clase',
       'Está ausente físicamente',
       'Está dando instrucciones para la próxima clase'],

docente_actividades = {
    "Instrucción docente": [
        {"full_name": "Está explicando conceptos de pensamiento computacional y/o programación", "clean_name": "Conceptos de PC"},
        {"full_name": "Está explicando conceptos relevantes para la lección, pero no asociados al pensamiento computacional o la programación", "clean_name": "Conceptos relevantes, no de PC"},
        {"full_name": "Está dando instrucciones para el desarrollo de una actividad", "clean_name": "Dar instrucciones"},
        {"full_name": "Está dando una explicación temática", "clean_name": "Explicar temas"},
        {"full_name": "Está copiando en el tablero sin hablar", "clean_name": "Copiar en tablero"},
    ],
    "Interacción": [
        {"full_name": "Está escuchando las intervenciones o presentaciones de los/las estudiantes", "clean_name": "Escuchar intervenciones"},
        {"full_name": "Está monitoreando o supervisando el trabajo de los/las estudiantes, sin intervenir", "clean_name": "Supervisar trabajos"},
        {"full_name": "Está haciendo preguntas a toda la clase", "clean_name": "Hacer preguntas"},
        {"full_name": "Está retroalimentando el trabajo realizado por los/las estudiantes (en parejas o grupos)", "clean_name": "Retroalimentar trabajos"},
        {"full_name": "Está aclarando dudas no relacionadas con instrucciones de actividades", "clean_name": "Aclarar dudas"},
    ],
    "Gestión de aula": [
        {"full_name": "Está haciendo actividades de gestión de aula (disciplina, atención de los/las estudiantes, etc)", "clean_name": "Gestión de aula"},
        {"full_name": "Está resolviendo problemas técnicos (uso del proyector, problemas con el computador, etc)", "clean_name": "Resolver problemas técnicos"},
    ],
    "Tareas no docente": [
        {"full_name": "Está ausente físicamente", "clean_name": "Ausente físicamente"},
        {"full_name": "Está presente físicamente, pero desconectado(a) de las actividades", "clean_name": "Desconectado de actividades"},
        {"full_name": "Está hablando de temas no relacionados con la clase", "clean_name": "Temas no relacionados"},
        {"full_name": "Está dando instrucciones para la próxima clase", "clean_name": "Instrucciones próxima clase"},
    ],
}

actividad_map = {x["full_name"]: {"clean": x["clean_name"], "cat": categoria} for categoria, valores in docente_actividades.items() for x in valores}


In [25]:
instantaneas

,id_respuesta,Número de instantánea,visita,doc_docente,nombre_docente,sexo_docente,guia_pedagogica,asignatura,Código IE,¿Qué está haciendo el/la docente ahora?,...,socializacion_quien,distraccion,no_involucrados,uso_tech,uso_tech_quien,trabajo_ind,trabajo_grupo,liderazgo,autonomia,comentarios
2,R_7esDIa291bLHEUw,1,2,3.379498e+06,Fabio Restrepo Restrepo,Hombre,Sí,Tecnología e Informática,NPC064,Está explicando conceptos de pensamiento compu...,...,NaN,Ninguno,NaN,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Todos(as),Ninguno,NaN
1,R_7r8EwEBEf2yeEhw,1,3,3.379498e+06,Fabio Restrepo Restrepo,Hombre,No,Tecnología e Informática,NPC064,Está explicando conceptos de pensamiento compu...,...,NaN,Uno o dos,Sólo los niños,Ninguno. No se están realizando actividades de...,Todos(as),Todos(as),Niños y niñas por igual,Todos(as),Ninguno,NaN
9,R_1ffrXqDCHFeK9vi,1,1,4.238721e+06,Luis Raúl Barón Manrique,Hombre,Sí,Tecnología e Informática,NPC110,Está explicando conceptos de pensamiento compu...,...,NaN,Uno o dos,Sólo los niños,Ninguno. No se están realizando actividades de...,Todos(as),Todos(as),Niños y niñas por igual,Todos(as),Ninguno,NaN
8,R_3SjC2EpHxhXK6yk,1,3,4.238721e+06,Luis Raúl Barón,Hombre,Sí,Tecnología e Informática,NPC110,Está explicando conceptos de pensamiento compu...,...,NaN,Ninguno,NaN,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Ninguno,Ninguno,NaN
11,R_5qjsHomeq9XN5F7,1,1,4.238845e+06,Hernando Diaz,Hombre,No,Tecnología e Informática,NPC234,Está haciendo preguntas a toda la clase,...,NaN,Pocos,Niños y niñas por igual,Ninguno. No se están realizando actividades de...,La mayoría,La mayoría,Niños y niñas por igual,Ninguno,Todos(as),Niños y niñas por igual
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,R_1ahGCVUVPVaxvGN,12,3,1.125763e+09,Yesenia Figueredo,Mujer,Sí,Tecnología e Informática,NPC071,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
800,R_7nIaUt1Pf0Z2r9l,12,1,1.127344e+09,José René Bernal Piza,Hombre,Sí,Tecnología e Informática,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
799,R_1efRKcyfddxQ0y0,12,3,1.127344e+09,José René Bernal Piza,Hombre,Sí,Tecnología e Informática,NPC324,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
806,R_7j6uWowM7Mi3LNj,12,1,1.130621e+09,Aleida Mabel Rosero Hernández,Mujer,Sí,Otro:,NPC222,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# Mapeo instantáneas
def map_actividad(actividad):
    if actividad in actividad_map:
        return actividad_map[actividad]["clean"], actividad_map[actividad]["cat"]
    else:
        return actividad, None

instantaneas[['accion_docente_clean', 'accion_docente_cat']] = instantaneas['¿Qué está haciendo el/la docente ahora?'].apply(lambda x: pd.Series(map_actividad(x)))


In [27]:
instantaneas

,id_respuesta,Número de instantánea,visita,doc_docente,nombre_docente,sexo_docente,guia_pedagogica,asignatura,Código IE,¿Qué está haciendo el/la docente ahora?,...,no_involucrados,uso_tech,uso_tech_quien,trabajo_ind,trabajo_grupo,liderazgo,autonomia,comentarios,accion_docente_clean,accion_docente_cat
2,R_7esDIa291bLHEUw,1,2,3.379498e+06,Fabio Restrepo Restrepo,Hombre,Sí,Tecnología e Informática,NPC064,Está explicando conceptos de pensamiento compu...,...,NaN,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Todos(as),Ninguno,NaN,Conceptos de PC,Instrucción docente
1,R_7r8EwEBEf2yeEhw,1,3,3.379498e+06,Fabio Restrepo Restrepo,Hombre,No,Tecnología e Informática,NPC064,Está explicando conceptos de pensamiento compu...,...,Sólo los niños,Ninguno. No se están realizando actividades de...,Todos(as),Todos(as),Niños y niñas por igual,Todos(as),Ninguno,NaN,Conceptos de PC,Instrucción docente
9,R_1ffrXqDCHFeK9vi,1,1,4.238721e+06,Luis Raúl Barón Manrique,Hombre,Sí,Tecnología e Informática,NPC110,Está explicando conceptos de pensamiento compu...,...,Sólo los niños,Ninguno. No se están realizando actividades de...,Todos(as),Todos(as),Niños y niñas por igual,Todos(as),Ninguno,NaN,Conceptos de PC,Instrucción docente
8,R_3SjC2EpHxhXK6yk,1,3,4.238721e+06,Luis Raúl Barón,Hombre,Sí,Tecnología e Informática,NPC110,Está explicando conceptos de pensamiento compu...,...,NaN,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Ninguno,Ninguno,NaN,Conceptos de PC,Instrucción docente
11,R_5qjsHomeq9XN5F7,1,1,4.238845e+06,Hernando Diaz,Hombre,No,Tecnología e Informática,NPC234,Está haciendo preguntas a toda la clase,...,Niños y niñas por igual,Ninguno. No se están realizando actividades de...,La mayoría,La mayoría,Niños y niñas por igual,Ninguno,Todos(as),Niños y niñas por igual,Hacer preguntas,Interacción
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,R_1ahGCVUVPVaxvGN,12,3,1.125763e+09,Yesenia Figueredo,Mujer,Sí,Tecnología e Informática,NPC071,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
800,R_7nIaUt1Pf0Z2r9l,12,1,1.127344e+09,José René Bernal Piza,Hombre,Sí,Tecnología e Informática,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
799,R_1efRKcyfddxQ0y0,12,3,1.127344e+09,José René Bernal Piza,Hombre,Sí,Tecnología e Informática,NPC324,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
806,R_7j6uWowM7Mi3LNj,12,1,1.130621e+09,Aleida Mabel Rosero Hernández,Mujer,Sí,Otro:,NPC222,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# Asignar 'momento' como 'Pre' para la primera visita y 'Post' para la segunda visita por cada docente
instantaneas = instantaneas.sort_values(['doc_docente', 'visita'])
instantaneas['visita_order'] = instantaneas.groupby('doc_docente')['visita'].rank(method='dense').astype('Int64')
instantaneas['momento'] = instantaneas['visita_order'].map({1: 'Pre', 2: 'Post'})
instantaneas = instantaneas.drop(columns=['visita_order'])
# Mostrar los primeros registros para verificar
instantaneas

,id_respuesta,Número de instantánea,visita,doc_docente,nombre_docente,sexo_docente,guia_pedagogica,asignatura,Código IE,¿Qué está haciendo el/la docente ahora?,...,uso_tech,uso_tech_quien,trabajo_ind,trabajo_grupo,liderazgo,autonomia,comentarios,accion_docente_clean,accion_docente_cat,momento
2,R_7esDIa291bLHEUw,1,2,3.379498e+06,Fabio Restrepo Restrepo,Hombre,Sí,Tecnología e Informática,NPC064,Está explicando conceptos de pensamiento compu...,...,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Todos(as),Ninguno,NaN,Conceptos de PC,Instrucción docente,Pre
2,R_7esDIa291bLHEUw,2,2,3.379498e+06,Fabio Restrepo Restrepo,Hombre,Sí,Tecnología e Informática,NPC064,Está explicando conceptos de pensamiento compu...,...,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Todos(as),Ninguno,NaN,Conceptos de PC,Instrucción docente,Pre
2,R_7esDIa291bLHEUw,3,2,3.379498e+06,Fabio Restrepo Restrepo,Hombre,Sí,Tecnología e Informática,NPC064,Está explicando conceptos de pensamiento compu...,...,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Todos(as),Ninguno,NaN,Conceptos de PC,Instrucción docente,Pre
2,R_7esDIa291bLHEUw,4,2,3.379498e+06,Fabio Restrepo Restrepo,Hombre,Sí,Tecnología e Informática,NPC064,Está dando instrucciones para el desarrollo de...,...,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Todos(as),Ninguno,NaN,Dar instrucciones,Instrucción docente,Pre
2,R_7esDIa291bLHEUw,5,2,3.379498e+06,Fabio Restrepo Restrepo,Hombre,Sí,Tecnología e Informática,NPC064,Está dando instrucciones para el desarrollo de...,...,Ninguno. No se están realizando actividades de...,Ninguno. No se están realizando actividades co...,Ninguno,NaN,Todos(as),Ninguno,NaN,Dar instrucciones,Instrucción docente,Pre
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,R_1xLytPcQt6uCjDV,8,3,1.130621e+09,Aleida Mabel Rosero Hernández,Mujer,Sí,Tecnología e Informática,NPC222,Está monitoreando o supervisando el trabajo de...,...,Ninguno. No se están realizando actividades de...,Todos(as),Todos(as),Niños y niñas por igual,Ninguno,Todos(as),Niños y niñas por igual,Supervisar trabajos,Interacción,Post
805,R_1xLytPcQt6uCjDV,9,3,1.130621e+09,Aleida Mabel Rosero Hernández,Mujer,Sí,Tecnología e Informática,NPC222,Está dando instrucciones para la próxima clase,...,Ninguno. No se están realizando actividades de...,Todos(as),Todos(as),Niños y niñas por igual,Ninguno,Todos(as),Niños y niñas por igual,Instrucciones próxima clase,Tareas no docente,Post
805,R_1xLytPcQt6uCjDV,10,3,1.130621e+09,Aleida Mabel Rosero Hernández,Mujer,Sí,Tecnología e Informática,NPC222,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Post
805,R_1xLytPcQt6uCjDV,11,3,1.130621e+09,Aleida Mabel Rosero Hernández,Mujer,Sí,Tecnología e Informática,NPC222,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Post


In [29]:
obs_generales[obs_generales['id_respuesta'] == 'R_1OXnE4tWiAVleTv']

,fecha_inicio,fecha_fin,visita,momento,fecha_acomp,fecha_registro,hora_inicio,duracion_clase,duracion_seg,doc_docente,...,equidad_corrige_sexismo,equidad_liderazgo_fem,equidad_acciones_afirm,equidad_reflexion,equidad_ninguna,equidad_otra,equidad_otra_detalle,info_adicional_equidad,metacognicion,uso_grafico_anclaje
15,2025-09-30 12:24:33,2025-09-30 12:43:01,3,Post,19/09/2025,2025-09-30 12:43:02.712,06:05,00:45,1107,5471999.0,...,Se corrigen comentarios y comportamientos sexi...,Se estimula el liderazgo femenino,Se realizan acciones afirmativas en términos d...,NaN,NaN,NaN,NaN,NaN,"Sí, se hacen preguntas de reflexión sobre el a...","Sí, se construye desde cero un gráfico tipo ma..."


In [30]:
import plotly.express as px


In [31]:
# Porcentaje de cada categoría por Momento e Instantánea, líneas que muestran el porcentaje de cada categoría para esa instantánea (cada instantánea suma 100%, el color de la línea es la categoría)
# Dividir en dos gráficos, uno para Pre y otro para Post

import plotly.express as px
import pandas as pd

# Assuming final_df_activities is already loaded
# Remove missing categories
df = instantaneas.dropna(subset=["accion_docente_cat"])

# Count how many occurrences of each category per instant and moment
df_counts = (
    df.groupby(["momento", "Número de instantánea", "accion_docente_cat"])
    .size()
    .reset_index(name="count")
)

# Calculate percentages per instant within each moment
df_counts["percent"] = (
    df_counts.groupby(["momento", "Número de instantánea"])["count"]
    .apply(lambda x: 100 * x / x.sum())
    .reset_index(drop=True)
)

# Split into two datasets
for momento in df_counts["momento"].unique():
    df_momento = df_counts[df_counts["momento"] == momento]

    fig = px.line(
        df_momento,
        x="Número de instantánea",
        y="percent",
        color="accion_docente_cat",
        markers=True,
        title=f"Distribución porcentual de categorías - {momento}",
        labels={"percent": "Porcentaje (%)", "Número de instantánea": "Número de instantánea"}
    )

    fig.update_layout(
        yaxis=dict(range=[0, 100]),
        legend_title_text="Categoría",
        template="plotly_white",
    )

    fig.show()



In [32]:
import pandas as pd
import plotly.express as px

# --- Prepare data ---
df_heat = instantaneas.copy()

# Ensure numeric and clean
df_heat['Número de instantánea'] = pd.to_numeric(df_heat['Número de instantánea'], errors='coerce')
df_heat = df_heat.dropna(subset=['Número de instantánea', 'accion_docente_clean'])

# Calculate percentages per instantánea and momento
df_counts = (
    df_heat.groupby(['momento', 'Número de instantánea', 'accion_docente_clean'])
    .size()
    .reset_index(name='count')
)

df_counts['percent'] = (
    df_counts.groupby(['momento', 'Número de instantánea'])['count']
    .apply(lambda x: 100 * x / x.sum())
    .reset_index(drop=True)
)

# Keep consistent y order by overall frequency
ordered_actions = df_counts['accion_docente_clean'].value_counts().index.tolist()
df_counts['accion_docente_clean'] = pd.Categorical(df_counts['accion_docente_clean'], categories=ordered_actions, ordered=True)

# --- Create pivot tables per momento ---
pivot_pre = df_counts[df_counts['momento'] == 'Pre'].pivot(index='accion_docente_clean', columns='Número de instantánea', values='percent')
pivot_post = df_counts[df_counts['momento'] == 'Post'].pivot(index='accion_docente_clean', columns='Número de instantánea', values='percent')

# --- Define same color scale as before (replace if needed) ---
color_scale = "Purples"

pivot_pre = pivot_pre.fillna(0)
pivot_post = pivot_post.fillna(0)

for momento in ["Pre", "Post"]:
    if momento == "Pre":
        data = pivot_pre
    else:
        data = pivot_post

    # --- Create side-by-side heatmaps (facets manually) ---
    fig = px.imshow(
        data,
        labels=dict(x="Número de instantánea", y="Acción del docente", color="Porcentaje"),
        x=data.columns,
        color_continuous_scale=color_scale,
        text_auto=".1f"
    )

    fig.update_layout(
        title=f"¿Qué está haciendo el docente? (Distribución porcentual por instantánea) - {momento}",
        height=850,
        margin=dict(l=280, r=40, t=80, b=40),
        coloraxis_colorbar=dict(title="Porcentaje"),
        template="plotly_white",
    )

    fig.update_traces(
        texttemplate="%{z:0.1f}%",
        hovertemplate="Acción: %{y}<br>Momento/Inst: %{x}<br>%{z:.1f}%<extra></extra>",
        textfont=dict(size=8),
    )

    fig.update_yaxes(autorange="reversed")

    fig.show()


In [33]:
obs_generales.to_csv('../data/limpieza/obs_generales_ti_limpio.csv', index=False)
instantaneas.to_csv('../data/limpieza/instantaneas_ti_limpio.csv', index=False)

In [34]:
import plotly.graph_objects as go


objetivos_map ={
    'objetivos_aprend': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Objetivos de Aprendizaje', 'category': 'Inicio de clase'},
    'conoc_previos': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Conocimientos Previos', 'category': 'Inicio de clase'},
    'conceptos_clave': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Conceptos Clave', 'category': 'Inicio de clase'},
    'vocabulario_comp': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Vocabulario adecuado', 'category': 'Conocimientos técnicos'},
    'conexion_vida': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Conexión con la vida diaria', 'category': 'Conocimientos técnicos'},
    'prep_material': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Preparación de materiales', 'category': 'Prácticas pedagógicas y de gestión de aula'},
    'gestion_material': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Gestión de materiales', 'category': 'Prácticas pedagógicas y de gestión de aula'},
    'valoracion_esfuerzo': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Valoración del esfuerzo', 'category': 'Prácticas pedagógicas y de gestión de aula'},
    'apoyo_estudiantes': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Apoyo a estudiantes', 'category': 'Prácticas pedagógicas y de gestión de aula'},
    'metacognicion': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Metacognición y reflexión', 'category': 'Cierre de clase'},
    'uso_grafico_anclaje': {'positive_words': ['sí'], 'negative_words': ['no'], 'clean_name': 'Uso de gráfico de anclaje o memoria colectiva', 'category': 'Cierre de clase'},
}

clean_objetivos = obs_generales.copy()

def clean_column_values(value, positive_words, negative_words):

    if pd.isna(value):
        return 'No'
    value = value.lower().strip()
    if any(pos_word in value for pos_word in positive_words):
        return 'Sí'
    if any(neg_word in value for neg_word in negative_words):
        return 'No'
    return 'No'


# clean column values
for col, mapping in objetivos_map.items():
    clean_objetivos[col] = clean_objetivos[col].str.lower().str.strip()
    clean_objetivos[col] = clean_objetivos [col].fillna('No')
    clean_objetivos[col] = clean_objetivos[col].apply(lambda x: clean_column_values(x, mapping['positive_words'], mapping['negative_words']))

In [35]:
for col, mapping in objetivos_map.items():
    for momento in ['Pre', 'Post']:
        mask = clean_objetivos['momento'] == momento
        pos_count = (clean_objetivos.loc[mask, col] == 'Sí').sum()
        neg_count = (clean_objetivos.loc[mask, col] == 'No').sum()
        total = pos_count + neg_count
        percentage = (pos_count / total * 100) if total > 0 else 0
        key = 'pre_percentage' if momento == 'Pre' else 'post_percentage'
        objetivos_map[col][key] = percentage

# Data grouped by section
# groups = {
#     'Inicio de clase': [
#         ('Objetivos de aprendizaje', 42, 89),
#         ('Conocimientos previos', 62, 88),
#         ('Conceptos clave', 70, 91)
#     ],
#     'Conocimientos técnicos': [
#         ('Vocabulario adecuado', 73, 91),
#         ('Conexión vida diaria', 42, 80)
#     ],
#     'Prácticas pedagógicas y de gestión de aula': [
#         ('Material preparado', 56, 86),
#         ('Gestión de materiales', 58, 82),
#         ('Valoración esfuerzo', 59, 80),
#         ('Estrategias de apoyo', 59, 74)
#     ],
#     'Cierre de la clase': [
#         ('Gráficos de anclaje', 8, 36),
#         ('Metacognición', 27, 74)
#     ]
# }

# Flatten data for plotting and record section starts
categories = []
pretest = []
posttest = []
section_starts = []
sections = []

# for section, items in groups.items():
#     section_starts.append(len(categories))
#     for cat, pre, post in items:
#         sections.append(section)
#         categories.append(cat)
#         pretest.append(pre)
#         posttest.append(post)


for col, mapping in objetivos_map.items():
    sections.append(mapping['category'])
    categories.append(mapping['clean_name'])
    pretest.append(mapping['pre_percentage'])
    posttest.append(mapping['post_percentage'])
    if mapping['category'] not in section_starts:
        section_starts.append(len(categories) - 1)



# Create figure
fig = go.Figure()

# Add connecting lines
for i in range(len(categories)):
    fig.add_trace(go.Scatter(
        x=[pretest[i], posttest[i]],
        y=[[sections[i], sections[i]], [categories[i], categories[i]]],
        mode='markers+lines',
        line=dict(color='lightgray', width=2),
        showlegend=False
    ))

full_sections = []

for section in sections:
    full_sections.append(section)
    full_sections.append(section)

# Add pretest and posttest markers + labels
fig.add_trace(go.Scatter(
    x=pretest, y=[sections, categories], mode='markers+text',
    name='Pretest', marker=dict(color='cornflowerblue', size=10),
    text=[f'{v:.1f}%' for v in pretest], textposition='middle left'
))

fig.add_trace(go.Scatter(
    x=posttest, y=[sections, categories], mode='markers+text',
    name='Postest', marker=dict(color='orchid', size=10),
    text=[f'{v:.1f}%' for v in posttest], textposition='middle right'
))


# Layout: reverse y to keep provided order top->bottom, increase left margin
fig.update_layout(
    title='Porcentaje de docentes observados en cada práctica',
    xaxis_title='Docentes observados',
    xaxis=dict(range=[0, 100], ticksuffix='%'),
    template='simple_white',
    height=700,
    yaxis_title='',
    yaxis=dict(autorange='reversed'),  # keep categories in the order we defined
    margin=dict(l=300, r=20, t=60, b=50),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()
